In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/bank-additional-full.csv', sep=';')

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  str    
 2   marital         41188 non-null  str    
 3   education       41188 non-null  str    
 4   default         41188 non-null  str    
 5   housing         41188 non-null  str    
 6   loan            41188 non-null  str    
 7   contact         41188 non-null  str    
 8   month           41188 non-null  str    
 9   day_of_week     41188 non-null  str    
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  str    
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null  float64
 1

In [3]:
df.nunique()

age                 78
job                 12
marital              4
education            8
default              3
housing              3
loan                 3
contact              2
month               10
day_of_week          5
duration          1544
campaign            42
pdays               27
previous             8
poutcome             3
emp.var.rate        10
cons.price.idx      26
cons.conf.idx       26
euribor3m          316
nr.employed         11
y                    2
dtype: int64

# 1. 标准处理

In [4]:
df = df.drop(columns=['duration'])

In [5]:
# 二分类变量直接二元编码
df[['y']] = df[['y']].replace({'yes': 1, 'no': 0}).astype(int)

In [6]:
# 多分类变量进行独热编码
df = pd.get_dummies(df, columns=['default', 'housing', 'loan', 'job', 'marital', 'education', 'poutcome', 'contact', 'month', 'day_of_week'], drop_first=True)

In [7]:
df.to_csv('data/processed_data_base1.csv', index=False)

# 2. 重点处理

In [8]:
df_test = df.copy()
df_test.head()

,age,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,...,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed
0,56,1,999,0,1.1,93.994,-36.4,4.857,5191.0,0,...,False,False,True,False,False,False,True,False,False,False
1,57,1,999,0,1.1,93.994,-36.4,4.857,5191.0,0,...,False,False,True,False,False,False,True,False,False,False
2,37,1,999,0,1.1,93.994,-36.4,4.857,5191.0,0,...,False,False,True,False,False,False,True,False,False,False
3,40,1,999,0,1.1,93.994,-36.4,4.857,5191.0,0,...,False,False,True,False,False,False,True,False,False,False
4,56,1,999,0,1.1,93.994,-36.4,4.857,5191.0,0,...,False,False,True,False,False,False,True,False,False,False


In [9]:
df_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 53 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   age                            41188 non-null  int64  
 1   campaign                       41188 non-null  int64  
 2   pdays                          41188 non-null  int64  
 3   previous                       41188 non-null  int64  
 4   emp.var.rate                   41188 non-null  float64
 5   cons.price.idx                 41188 non-null  float64
 6   cons.conf.idx                  41188 non-null  float64
 7   euribor3m                      41188 non-null  float64
 8   nr.employed                    41188 non-null  float64
 9   y                              41188 non-null  int64  
 10  default_unknown                41188 non-null  bool   
 11  default_yes                    41188 non-null  bool   
 12  housing_unknown                41188 non-null  bool   
 1

1. 对 campaign、previous 尝试对数变换。

In [10]:
df_test['campaign'] = np.log1p(df_test['campaign'])
df_test['previous'] = np.log1p(df_test['previous'])

2. 对于pdays，区分客户以前是否被联系过，以及联系过的客户距离上次联系有多少天。由于未联系过的占大多数，所以仅保留是否联系过


In [11]:
df_test['pre_contact'] = (df_test['pdays'] != 999).astype(int)
df_test.drop(columns=['pdays'], inplace=True)

In [12]:
df_test['pre_contact'].value_counts()

pre_contact
0    39673
1     1515
Name: count, dtype: int64

3. 对于年龄可以考虑分箱

In [13]:
df_test['age_group'] = pd.cut(df_test['age'], bins=[0, 25, 35, 45, 55, 65, 100], labels=['0-25', '26-35', '36-45', '46-55', '56-65', '65+'])
df_test.drop(columns=['age'], inplace=True)
df_test = pd.get_dummies(df_test, columns=['age_group'], drop_first=True)

In [14]:
df['y'].value_counts()

y
0    36548
1     4640
Name: count, dtype: int64

In [15]:
df_test.to_csv('data/processed_data1.csv', index=False)